## 1. Setup

### Domain-Driven Feature Engineering Philosophy

Raw ICU data contains many individual measurements, but clinical knowledge tells us
that **composite indices** often capture physiological state better than individual
variables. For example:

- **Shock Index** (heart rate / systolic BP) is a validated triage tool — values > 1.0
  indicate haemodynamic instability with much higher mortality risk than either
  variable alone.
- **BMI category** (underweight / normal / overweight / obese) captures non-linear
  associations between body mass and ICU outcomes.
- **Pulse Pressure** (systolic − diastolic BP) reflects arterial stiffness and
  cardiovascular strain.
- **Age × APACHE interaction** captures the compounding effect of age on severity.

This notebook applies the `build_features` pipeline from `src/features.py` and
validates the resulting features with clinical sanity checks and target correlation
analysis. The goal is to produce a feature matrix ready for model training.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import load_raw, drop_high_missingness, impute

os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Load and preprocess raw data
df_raw = load_raw('../data/raw/patient_survival.csv')
print(f'Raw data shape: {df_raw.shape}')

df_clean = drop_high_missingness(df_raw, threshold=0.6)
print(f'After dropping high-missingness columns: {df_clean.shape}')

df_imputed = impute(df_clean)
print(f'After imputation: {df_imputed.shape}')
print(f'Remaining nulls: {df_imputed.isnull().sum().sum()}')

## 2. Apply Feature Engineering Pipeline

The `build_features` function in `src/features.py` applies the full domain-driven
transformation pipeline. It creates composite clinical indices, bins continuous
variables into clinically meaningful categories, and encodes categorical features.
We track the shape change and list the new columns added.

In [ ]:
from src.features import build_features

cols_before = set(df_imputed.columns)
print(f'Shape before build_features: {df_imputed.shape}')

df_engineered = build_features(df_imputed)
print(f'Shape after build_features:  {df_engineered.shape}')

cols_after = set(df_engineered.columns)
new_cols = sorted(cols_after - cols_before)
dropped_cols = sorted(cols_before - cols_after)

print(f'\nNew columns added ({len(new_cols)}):')
for c in new_cols:
    print(f'  + {c}')

if dropped_cols:
    print(f'\nColumns dropped ({len(dropped_cols)}):')
    for c in dropped_cols:
        print(f'  - {c}')

print(f'\nSample of engineered dataframe:')
display_cols = ['hospital_death'] + new_cols[:5] if new_cols else list(df_engineered.columns[:6])
df_engineered[display_cols].head()

## 3. Validate New Features — Clinical Sanity Checks

Before trusting engineered features in a model, we perform clinical sanity checks:

- **Shock Index** should be positively associated with mortality (sicker patients
  have higher values). Distributions should differ clearly between survived/died groups.
- **BMI Category** has a known U-shaped relationship with ICU mortality —
  underweight and extremely obese patients are at higher risk.

These plots serve as a **data contract test**: if the expected clinical relationship
is absent, there is likely a data quality or transformation bug.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Shock Index histogram by outcome ---
shock_col = 'shock_index' if 'shock_index' in df_engineered.columns else None
if shock_col:
    for outcome, color, label in [(0, '#4878CF', 'Survived'), (1, '#D65F5F', 'Died')]:
        vals = df_engineered.loc[df_engineered['hospital_death'] == outcome, shock_col].dropna()
        vals_clipped = vals.clip(0, 3)
        axes[0].hist(vals_clipped, bins=60, alpha=0.55, color=color, label=label, density=True)
    axes[0].set_title('Shock Index Distribution by Outcome', fontweight='bold')
    axes[0].set_xlabel('Shock Index (HR / Systolic BP)')
    axes[0].set_ylabel('Density')
    axes[0].axvline(1.0, color='black', linestyle='--', linewidth=1.2, label='SI = 1.0 threshold')
    axes[0].legend()
else:
    # Fallback: use any numeric engineered feature
    feat = new_cols[0] if new_cols else df_engineered.select_dtypes(include=np.number).columns[1]
    for outcome, color, label in [(0, '#4878CF', 'Survived'), (1, '#D65F5F', 'Died')]:
        vals = df_engineered.loc[df_engineered['hospital_death'] == outcome, feat].dropna()
        axes[0].hist(vals, bins=50, alpha=0.55, color=color, label=label, density=True)
    axes[0].set_title(f'{feat} by Outcome', fontweight='bold')
    axes[0].set_xlabel(feat)
    axes[0].set_ylabel('Density')
    axes[0].legend()

# --- BMI category mortality rate crosstab ---
bmi_cat_col = 'bmi_category' if 'bmi_category' in df_engineered.columns else None
if bmi_cat_col:
    ct = pd.crosstab(df_engineered[bmi_cat_col], df_engineered['hospital_death'], normalize='index') * 100
    if 1 in ct.columns:
        mort_by_bmi = ct[1].sort_values(ascending=False)
        axes[1].bar(mort_by_bmi.index.astype(str), mort_by_bmi.values, color='#D65F5F', alpha=0.75, edgecolor='white')
        axes[1].set_title('Mortality Rate by BMI Category', fontweight='bold')
        axes[1].set_xlabel('BMI Category')
        axes[1].set_ylabel('Mortality Rate (%)')
        for i, val in enumerate(mort_by_bmi.values):
            axes[1].text(i, val + 0.2, f'{val:.1f}%%', ha='center', fontsize=10)
else:
    # Fallback: age group mortality rate
    df_engineered['age_bin_tmp'] = pd.cut(df_engineered['age'], bins=[0, 40, 60, 75, 120],
                                           labels=['18-40', '40-60', '60-75', '75+'])
    ct = df_engineered.groupby('age_bin_tmp')['hospital_death'].mean() * 100
    axes[1].bar(ct.index.astype(str), ct.values, color='#D65F5F', alpha=0.75, edgecolor='white')
    axes[1].set_title('Mortality Rate by Age Group', fontweight='bold')
    axes[1].set_xlabel('Age Group')
    axes[1].set_ylabel('Mortality Rate (%)')

plt.suptitle('Engineered Feature Validation — Clinical Sanity Checks', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/engineered_features_validation.png', bbox_inches='tight')
plt.show()

## 4. Feature Correlation with Target

Point-biserial correlation quantifies the linear association between each numeric
feature and the binary `hospital_death` target. While models like LightGBM can
capture non-linear associations, this analysis helps identify features with
**strong marginal signal** and confirms that our engineered features add
discriminative power beyond the raw variables.

In [ ]:
from scipy import stats

numeric_cols = df_engineered.select_dtypes(include=[np.number]).columns.tolist()
if 'hospital_death' in numeric_cols:
    numeric_cols.remove('hospital_death')

pb_corrs = {}
y = df_engineered['hospital_death'].values

for col in numeric_cols:
    x = df_engineered[col].fillna(df_engineered[col].median()).values
    if np.std(x) > 0:
        corr, pval = stats.pointbiserialr(y, x)
        pb_corrs[col] = corr

pb_series = pd.Series(pb_corrs).sort_values(key=abs, ascending=False)
top20 = pb_series.head(20)

print('Top 20 features by absolute point-biserial correlation with hospital_death:')
print(top20.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#D65F5F' if v > 0 else '#4878CF' for v in top20.values]
ax.barh(range(len(top20)), top20.values, color=colors, edgecolor='white', alpha=0.85)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Point-Biserial Correlation with hospital_death')
ax.set_title('Top 20 Features by Correlation with Mortality Target', fontsize=12, fontweight='bold')

for i, val in enumerate(top20.values):
    ax.text(val + (0.002 if val >= 0 else -0.002), i,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/figures/feature_target_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
output_path = '../data/processed/features_engineered.csv'
df_engineered.to_csv(output_path, index=False)
print(f'Engineered features saved to: {output_path}')
print(f'Final shape: {df_engineered.shape}')
print(f'\nColumn summary:')
print(f'  Numeric columns:     {df_engineered.select_dtypes(include=np.number).shape[1]}')
print(f'  Categorical columns: {df_engineered.select_dtypes(exclude=np.number).shape[1]}')
print(f'  Target (hospital_death) mean: {df_engineered["hospital_death"].mean():.4f}')